In [ ]:
# Progress heartbeat: every 5 min during the first hour, then every 30 min, a tiny version of the
# private dataset cv5fold-progress with what the notebook is doing (current line, epochs, last log
# lines, GPU use). Kaggle shows no log during a committed run, so this dataset is used to follow its
# progress. It never touches the runs or the store.
import json, os, socket, subprocess, sys, threading, time, traceback
from pathlib import Path

socket.setdefaulttimeout(120)  # a stalled download fails after 2 min instead of hanging the session

from kaggle_secrets import UserSecretsClient
_hb_secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = _hb_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = _hb_secrets.get_secret("KAGGLE_KEY")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.2"])

HB_ID = f"{os.environ['KAGGLE_USERNAME']}/cv5fold-progress"
HB_STAGE = Path("/tmp/heartbeat_stage")
HB_RUNS = Path("/kaggle/working")
HB_T0 = time.time()
HB_MAIN = threading.main_thread().ident


def _hb_tail(path, n=3):
    try:
        text = path.read_bytes()[-20000:].decode(errors="replace")
    except OSError:
        return []
    lines = [l.split("\r")[-1].strip() for l in text.split("\n")]  # last state of progress bars
    return [l[:220] for l in lines if l][-n:]


def _hb_report():
    frame = sys._current_frames().get(HB_MAIN)
    stack = traceback.extract_stack(frame) if frame is not None else []
    cell = [f for f in stack if "ipykernel" in f.filename or f.filename.startswith("<ipython")]
    where = f"cell line {cell[-1].lineno}: {cell[-1].line}" if cell else "idle (between cells or finished)"
    hours = (time.time() - HB_T0) / 3600
    lines = [f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {hours:.2f} h since the heartbeat started",
             f"notebook is at: {where}", "", "stack of the notebook (last 15 frames):"]
    lines += [f"  {Path(f.filename).name}:{f.lineno} {f.name} | {(f.line or '')[:150]}" for f in stack[-15:]]
    running = []
    for d in sorted(HB_RUNS.glob("cv_*/*/test*_seed*")):
        csv_path = d / "train" / "results.csv"
        n = max(0, len(csv_path.read_text().splitlines()) - 1) if csv_path.exists() else 0
        done = (d / "summary.json").exists()
        if not done:
            running.append(f"{d.name.split('_')[0]} ep {n}")
        lines += ["", f"{d.parent.name}/{d.name}: {'done' if done else f'epoch {n}'}"]
        lines += [f"  log: {l}" for l in _hb_tail(d.parent / f"log_{d.name}.txt")]
    try:
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip().replace("\n", " ; ")
    except Exception as e:
        gpu = f"n/a ({e.__class__.__name__})"
    lines += ["", f"GPU (utilization, memory): {gpu}"]
    state = ", ".join(running) if running else ((cell[-1].line or "").strip()[:70] if cell else "idle")
    return "\n".join(lines), f"{hours:.2f} h | {state} | GPU {gpu}"[:190]


def _hb_upload(text, note):
    HB_STAGE.mkdir(parents=True, exist_ok=True)
    (HB_STAGE / "progress.txt").write_text(text)
    (HB_STAGE / "dataset-metadata.json").write_text(json.dumps(
        {"title": "cv5fold-progress", "id": HB_ID, "licenses": [{"name": "CC0-1.0"}]}))
    r = subprocess.run(["kaggle", "datasets", "status", HB_ID], capture_output=True, text=True, timeout=120)
    low = (r.stdout + r.stderr).lower()
    exists = r.returncode == 0 and not any(s in low for s in ("404", "not found", "forbidden"))
    args = ["version", "-p", str(HB_STAGE), "-m", note] if exists else ["create", "-p", str(HB_STAGE)]
    subprocess.run(["kaggle", "datasets", *args, "-q"], capture_output=True, text=True, timeout=600)


def _hb_loop():
    while True:
        try:
            _hb_upload(*_hb_report())
        except Exception:
            pass  # the heartbeat must never disturb the notebook
        time.sleep(300 if time.time() - HB_T0 < 3600 else 1800)


threading.Thread(target=_hb_loop, daemon=True, name="heartbeat").start()
print("Heartbeat started: follow the session in the dataset", HB_ID)


# Patient-level 5-fold cross-validation — component ablations

Same pipeline as `cv5fold_patient_level_multi.ipynb` (data, folds, protocol, store, archive), applied to three single-component variants of the adopted model: `a2conv` (area attention of the A2C2f block replaced by C3k convolutions), `yolo12neck` (YOLOv12n neck instead of the SSDLiteX-inspired neck), and `noadaptor` (BatchNorm input adaptor removed). The variant is selected by `CONFIG` in section 1, built in section 5, and checked in section 6. All variants use full fine-tuning, as the adopted model.

For each official Figshare fold *k*: **test = fold k** (never seen during training or selection),
**validation = fold k+1** (early stopping and checkpoint selection only), **training = the three other folds**.
The optimization protocol is identical to the public scripts (repository v1.0.0); only the split changes.

**Kaggle settings:** input dataset `figshare-yolo-trainval-official` (it contains `split_metadata.csv`),
accelerator **GPU T4 x2**, **Internet on**, and the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`
(*Add-ons → Secrets*, attached to this notebook).

**Persistent store.** Every `SNAPSHOT_MINUTES` and whenever a fold finishes, the runs are uploaded as a new
version of a private Kaggle dataset (`cv5fold-runs-<config>-seed<seed>`). At the start of each
session the latest version is restored: completed folds are skipped, interrupted folds resume from their last
checkpoint. To continue after an interruption, simply run the same notebook again.

1. Smoke test (GPU, about 10 min): `SMOKE_TEST = True`, *Run All*. It checks the whole pipeline (4 epochs per
   fold), times one epoch, interrupts fold 1 on purpose after 2 epochs and resumes it, and uploads to a separate
   dataset (`cv5fold-smoke-…`, which can be deleted afterwards).
   Store self-test (Accelerator *None*, no GPU quota): `STORE_SELF_TEST = True`, *Run All*. It uploads a small run
   tree, restores it file by file and compares (`cv5fold-storetest`, which can be deleted afterwards).
2. Then `SMOKE_TEST = False`, choose `TEST_FOLDS`, *Save Version → Save & Run All (Commit)*.
3. Results are in the store dataset and in the `.zip` of the Output tab.

## 1. Configuration

In [ ]:
CONFIG = "a2conv"    # ablations: "a2conv" = area attention of A2C2f replaced by C3k convolutions
                     # | "yolo12neck" = YOLOv12n neck instead of the SSDLiteX-inspired neck
                     # | "noadaptor" = BatchNorm input adaptor removed
                     # ("adopted", "yolov12n" and "frozen" are also accepted)
TEST_FOLDS = [1, 2]  # test folds of this session; with T4 x2, two folds run in parallel (one per GPU)
SMOKE_TEST = True    # True: 4 epochs on folds 1 (and 2): pipeline, timing, resumption and store test
SEED = 42
ULTRALYTICS_VERSION = "8.4.104"
USE_STORE = True       # back up runs to a private Kaggle dataset and resume from it
SNAPSHOT_MINUTES = 30  # maximum computation lost if the session is interrupted
STORE_SELF_TEST = False  # True with Accelerator "None": test the store round trip without GPU, no training

from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

## 2. Environment

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"])

import json, os, platform, shutil, time
import pandas as pd, torch, torchvision, ultralytics, yaml

assert ultralytics.__version__ == ULTRALYTICS_VERSION, ultralytics.__version__
N_GPU = torch.cuda.device_count()
ENV = {
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "ultralytics": ultralytics.__version__, "cuda": torch.version.cuda, "n_gpu": N_GPU,
    "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "cpu_count": os.cpu_count(),
}
print(json.dumps(ENV, indent=2))
assert N_GPU >= 1 or STORE_SELF_TEST, "Enable a GPU accelerator (GPU T4 x2 recommended)."

## 3. Dataset and official folds — integrity checks

In [ ]:
hits = sorted(KAGGLE_INPUT.rglob("split_metadata.csv"))
assert len(hits) == 1, f"Expected one split_metadata.csv under {KAGGLE_INPUT}, found: {hits}"
SRC = hits[0].parent
md = pd.read_csv(hits[0])
md["name"] = md["image"].map(lambda s: Path(s).name)
md["img_src"] = [SRC / s / "images" / n for s, n in zip(md["split"], md["name"])]
md["lab_src"] = [SRC / s / "labels" / Path(f).name for s, f in zip(md["split"], md["label_file"])]

# Expected (meningioma, glioma, pituitary) slices per official fold
EXPECTED = {1: (112, 245, 185), 2: (167, 338, 174), 3: (139, 231, 202), 4: (124, 325, 179), 5: (166, 287, 190)}
assert len(md) == 3064 and md["name"].is_unique
assert md["pid"].nunique() == 233
assert (md.groupby("pid")["fold"].nunique() == 1).all(), "A patient appears in more than one fold."
for k, expected in EXPECTED.items():
    got = tuple(int(((md["fold"] == k) & (md["class_id"] == c)).sum()) for c in range(3))
    assert got == expected, (k, got, expected)
for r in md.itertuples():
    assert r.img_src.is_file() and r.lab_src.is_file(), r.name
    lines = [l.split() for l in r.lab_src.read_text().splitlines() if l.strip()]
    assert len(lines) == 1 and len(lines[0]) == 5 and int(lines[0][0]) == r.class_id, r.lab_src

print("Dataset:", SRC)
print(md.groupby("fold").agg(images=("name", "size"), patients=("pid", "nunique")).T)
print("All checks passed: 3,064 slices, 233 patients, no patient shared between folds, labels consistent.")

## 4. Local copy and per-fold partitions

In [ ]:
POOL = WORK / "pool"
for sub in ("images", "labels"):
    (POOL / sub).mkdir(parents=True, exist_ok=True)
for r in md.itertuples():
    for src, sub in ((r.img_src, "images"), (r.lab_src, "labels")):
        dst = POOL / sub / src.name
        if not dst.exists():
            shutil.copy2(src, dst)

NAMES = {0: "meningioma", 1: "glioma", 2: "pituitary_tumor"}
CV_DATA = WORK / "cv_data"


def build_fold(k):
    # test = fold k, val = fold k+1 (cyclic), train = the three remaining folds
    v = k % 5 + 1
    roles = {"train": [f for f in range(1, 6) if f not in (k, v)], "val": [v], "test": [k]}
    base = CV_DATA / f"test{k}"
    if base.exists():
        shutil.rmtree(base)
    rows = []
    for role, folds in roles.items():
        for sub in ("images", "labels"):
            (base / role / sub).mkdir(parents=True)
        for r in md[md["fold"].isin(folds)].itertuples():
            os.symlink(POOL / "images" / r.name, base / role / "images" / r.name)
            os.symlink(POOL / "labels" / r.lab_src.name, base / role / "labels" / r.lab_src.name)
            rows.append((r.name, str(r.pid), int(r.class_id), int(r.fold), role))
    manifest = pd.DataFrame(rows, columns=["image", "pid", "class_id", "fold", "role"])
    pids = {role: set(manifest.loc[manifest["role"] == role, "pid"]) for role in roles}
    assert not (pids["train"] & pids["val"] or pids["train"] & pids["test"] or pids["val"] & pids["test"])
    manifest.to_csv(base / "fold_manifest.csv", index=False)
    data_yaml = base / "data.yaml"
    data_yaml.write_text(yaml.safe_dump(
        {"path": str(base), "train": "train/images", "val": "val/images", "test": "test/images",
         "nc": 3, "names": NAMES}, sort_keys=False))
    info = {"test_fold": k, "val_fold": v, "train_folds": roles["train"]}
    for role in roles:
        info[f"n_{role}_images"] = int((manifest["role"] == role).sum())
        info[f"n_{role}_patients"] = len(pids[role])
    info_path = base / "fold_info.json"
    info_path.write_text(json.dumps(info, indent=2))
    return data_yaml, info_path, info


for k in range(1, 6):
    v = k % 5 + 1
    tr = [f for f in range(1, 6) if f not in (k, v)]
    sub = lambda fs: md[md["fold"].isin(fs)]
    print(f"test fold {k}: {len(sub([k]))} img / {sub([k])['pid'].nunique()} pts | "
          f"val fold {v}: {len(sub([v]))} / {sub([v])['pid'].nunique()} | "
          f"train {tr}: {len(sub(tr))} / {sub(tr)['pid'].nunique()}")

## 5. Model definition and training script

In [ ]:
ARCH_YAML_TEXT = r'''
# Lightweight YOLOv12 detector used in the study.
# Ultralytics 8.4.104 configuration.
#
# Architecture:
# - learnable 3-channel BatchNorm input adaptor
# - truncated ImageNet-pretrained MobileNetV3-Small backbone
# - depthwise-separable SSDLiteX-inspired FPN-PAN neck
# - one A2C2f block on the P5 semantic path
# - three-scale anchor-free Detect head

nc: 3
reg_max: 16
scale: n

backbone:
  # [from, repeats, module, arguments]
  - [-1, 1, nn.BatchNorm2d, []]
  - [-1, 1, TorchVision, [96, mobilenet_v3_small, DEFAULT, true, 5, true]]
  - [1, 1, Index, [24, 4]]
  - [1, 1, Index, [48, 9]]
  - [1, 1, Index, [96, 10]]

head:
  # P5 semantic processing
  - [4, 1, Conv, [96, 1, 1]]
  - [-1, 1, DWConv, [96, 5, 1]]
  - [-1, 1, Conv, [96, 1, 1]]
  - [-1, 1, A2C2f, [96, true, 1, false, 2.0, 0.6666667]]

  # Top-down P5 -> P4
  - [-1, 1, nn.Upsample, [null, 2, nearest]]
  - [3, 1, Conv, [64, 1, 1]]
  - [[-2, -1], 1, Concat, [1]]
  - [-1, 1, Conv, [64, 1, 1]]
  - [-1, 1, DWConv, [64, 3, 1]]
  - [-1, 1, Conv, [64, 1, 1]]

  # Top-down P4 -> P3
  - [-1, 1, nn.Upsample, [null, 2, nearest]]
  - [2, 1, Conv, [48, 1, 1]]
  - [[-2, -1], 1, Concat, [1]]
  - [-1, 1, Conv, [48, 1, 1]]
  - [-1, 1, DWConv, [48, 3, 1]]
  - [-1, 1, Conv, [48, 1, 1]]

  # Highest-resolution P3 refinement
  - [-1, 1, DWConv, [48, 3, 1]]
  - [-1, 1, Conv, [48, 1, 1]]

  # Bottom-up P3 -> P4
  - [-1, 1, DWConv, [48, 3, 2]]
  - [-1, 1, Conv, [64, 1, 1]]
  - [[-1, 14], 1, Concat, [1]]
  - [-1, 1, Conv, [64, 1, 1]]
  - [-1, 1, DWConv, [64, 3, 1]]
  - [-1, 1, Conv, [64, 1, 1]]

  # Bottom-up P4 -> P5
  - [-1, 1, DWConv, [64, 3, 2]]
  - [-1, 1, Conv, [96, 1, 1]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 1, Conv, [96, 1, 1]]
  - [-1, 1, DWConv, [96, 3, 1]]
  - [-1, 1, Conv, [96, 1, 1]]

  # Three-scale detection head: P3, P4, P5
  - [[22, 28, 34], 1, Detect, [nc]]
'''[1:]  # exact copy of models/yolo12n_mnv3s_ssdlitex_lite.yaml

import hashlib
MODEL_YAML = WORK / "yolo12n_mnv3s_ssdlitex_lite.yaml"
MODEL_YAML.write_text(ARCH_YAML_TEXT, encoding="utf-8")
digest = hashlib.sha256(MODEL_YAML.read_bytes()).hexdigest()
assert digest == "63ead951033bdad7f92e2d9f6309fc96290ef8a126c466da711726dba497296f", "Architecture YAML differs from repository v1.0.0"
print("Architecture YAML identical to repository v1.0.0 (sha256 %s)" % digest[:16])

# Component ablations: each variant = the verified YAML above with exact text replacements.
# The file name keeps the prefix "yolo12n": Ultralytics 8.4.104 reads the model scale ("n") from it,
# as for the adopted model; without it A2C2f receives two extra arguments and "a2conv" fails.
ABLATIONS = {
    "a2conv": [("  - [-1, 1, A2C2f, [96, true, 1, false, 2.0, 0.6666667]]\n",
                "  - [-1, 1, A2C2f, [96, false, 1, false, 2.0, 0.6666667]]  # ablation: area attention -> C3k convolutions\n")],
    "noadaptor": [("  - [-1, 1, nn.BatchNorm2d, []]\n",
                   "  - [-1, 1, nn.Identity, []]  # ablation: BatchNorm input adaptor removed\n")],
    "yolo12neck": [('  # Top-down P5 -> P4\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [3, 1, Conv, [64, 1, 1]]\n  - [[-2, -1], 1, Concat, [1]]\n  - [-1, 1, Conv, [64, 1, 1]]\n  - [-1, 1, DWConv, [64, 3, 1]]\n  - [-1, 1, Conv, [64, 1, 1]]\n\n  # Top-down P4 -> P3\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [2, 1, Conv, [48, 1, 1]]\n  - [[-2, -1], 1, Concat, [1]]\n  - [-1, 1, Conv, [48, 1, 1]]\n  - [-1, 1, DWConv, [48, 3, 1]]\n  - [-1, 1, Conv, [48, 1, 1]]\n\n  # Highest-resolution P3 refinement\n  - [-1, 1, DWConv, [48, 3, 1]]\n  - [-1, 1, Conv, [48, 1, 1]]\n\n  # Bottom-up P3 -> P4\n  - [-1, 1, DWConv, [48, 3, 2]]\n  - [-1, 1, Conv, [64, 1, 1]]\n  - [[-1, 14], 1, Concat, [1]]\n  - [-1, 1, Conv, [64, 1, 1]]\n  - [-1, 1, DWConv, [64, 3, 1]]\n  - [-1, 1, Conv, [64, 1, 1]]\n\n  # Bottom-up P4 -> P5\n  - [-1, 1, DWConv, [64, 3, 2]]\n  - [-1, 1, Conv, [96, 1, 1]]\n  - [[-1, 8], 1, Concat, [1]]\n  - [-1, 1, Conv, [96, 1, 1]]\n  - [-1, 1, DWConv, [96, 3, 1]]\n  - [-1, 1, Conv, [96, 1, 1]]\n\n  # Three-scale detection head: P3, P4, P5\n  - [[22, 28, 34], 1, Detect, [nc]]\n',
                    '  # ablation: YOLOv12n neck (head of yolo12.yaml at scale n, channels written explicitly)\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 3], 1, Concat, [1]]\n  - [-1, 1, A2C2f, [128, false, -1]]\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 2], 1, Concat, [1]]\n  - [-1, 1, A2C2f, [64, false, -1]]\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [[-1, 11], 1, Concat, [1]]\n  - [-1, 1, A2C2f, [128, false, -1]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [[-1, 8], 1, Concat, [1]]\n  - [-1, 1, C3k2, [256, true]]\n\n  # Three-scale detection head: P3, P4, P5\n  - [[14, 17, 20], 1, Detect, [nc]]\n')],
}
# Expected unfused parameters (model, detection head); adopted model: 746,015 and 265,449.
EXPECTED = {"a2conv": (717_727, 265_449), "yolo12neck": (1_517_119, 431_257), "noadaptor": (746_009, 265_449)}
if CONFIG in ABLATIONS:
    variant = ARCH_YAML_TEXT
    for old, new in ABLATIONS[CONFIG]:
        assert variant.count(old) == 1, (CONFIG, old[:60])
        variant = variant.replace(old, new)
    MODEL_YAML = WORK / f"yolo12n_mnv3s_ssdlitex_lite_{CONFIG}.yaml"
    MODEL_YAML.write_text(variant, encoding="utf-8")
    print(f"Ablation {CONFIG}: {MODEL_YAML.name} (sha256 {hashlib.sha256(MODEL_YAML.read_bytes()).hexdigest()[:16]})")


In [ ]:
%%writefile /kaggle/working/train_fold.py
"""Train one configuration on one patient-level cross-validation fold, then
evaluate the selected checkpoint on the held-out test fold.

The optimization protocol is identical to training_arguments() in the public
scripts (repository v1.0.0). Only the data split changes:
  train = three official folds,
  val   = fold k+1 (early stopping and checkpoint selection only),
  test  = fold k (never seen during training or selection).

With --resume, an interrupted run continues from train/weights/last.pt. Ultralytics
restores weights, EMA, optimizer, AMP scaler, learning-rate schedule and best fitness;
the early-stopping counter is restored here from results.csv, because Ultralytics
8.4.104 recreates it empty on resume. Random-number-generator states are not stored
in checkpoints, so a resumed run is not bit-identical to an uninterrupted one; every
resumption is recorded in resume_log.json and in summary.json.
"""

from __future__ import annotations

import argparse
import csv
import json
import platform
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

# Matched protocol: same values as training_arguments() in scripts/train_*.py.
PROTOCOL: dict[str, object] = {
    "epochs": 400,
    "patience": 100,
    "imgsz": 640,
    "batch": 8,
    "workers": 2,
    "amp": True,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cos_lr": True,
    "deterministic": True,
    "pretrained": True,
    "cls_remap": True,
    "cache": False,
    "rect": False,
    "close_mosaic": 10,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "iou": 0.7,
    "max_det": 300,
    "save": True,
    "save_period": 25,
    "val": True,
    "plots": True,
    "verbose": True,
}

# freeze=2 freezes top-level modules 0 and 1 (input adaptor and backbone).
FREEZE: dict[str, int | None] = {"adopted": None, "frozen": 2, "yolov12n": None,
                                 "a2conv": None, "yolo12neck": None, "noadaptor": None}  # ablation variants

# Test-time evaluation: Ultralytics validator defaults, stated explicitly.
TEST_EVAL: dict[str, object] = {
    "imgsz": 640,
    "batch": 16,
    "conf": 0.001,
    "iou": 0.7,
    "max_det": 300,
    "save_json": True,
    "plots": True,
    "workers": 2,
    "verbose": True,
}

FITNESS_KEY = "metrics/mAP50-95(B)"  # sole fitness criterion in Ultralytics 8.4.104


def now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def read_results_csv(path: Path) -> dict[str, list[float]]:
    """Read an Ultralytics results.csv into numeric columns (header names stripped)."""
    with open(path, newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.reader(handle) if row]
    header = [name.strip() for name in rows[0]]
    columns: dict[str, list[float]] = {name: [] for name in header}
    for row in rows[1:]:
        for name, value in zip(header, row):
            columns[name].append(float(value))
    return columns


def truncate_results_csv(path: Path, last_epoch: int) -> int:
    """Keep complete rows up to last_epoch (1-based), the epoch stored in last.pt.

    results.csv is written just before last.pt at every epoch, so a copy taken
    between the two writes can be one row ahead; an interrupted write can also
    leave a partial last line.
    """
    lines = path.read_text(encoding="utf-8").splitlines()
    header, kept = lines[0], [lines[0]]
    n_cols = len(header.split(","))
    for line in lines[1:]:
        fields = line.split(",")
        try:
            values = [float(v) for v in fields]
        except ValueError:
            continue
        if len(fields) == n_cols and values[0] <= last_epoch:
            kept.append(line)
    path.write_text("\n".join(kept) + "\n", encoding="utf-8")
    return len(kept) - 1


def selection_info(results: dict[str, list[float]]) -> dict[str, float | int]:
    """Locate the epoch whose checkpoint Ultralytics kept as best.pt.

    best.pt is replaced only on strict improvement of the fitness (validation
    mAP50-95), so best.pt is the first maximum.
    """
    fitness = results[FITNESS_KEY]
    index = int(np.argmax(fitness))
    info: dict[str, float | int] = {
        "best_epoch": int(results["epoch"][index]),
        "epochs_completed": int(results["epoch"][-1]),
        "val_precision_at_best": results["metrics/precision(B)"][index],
        "val_recall_at_best": results["metrics/recall(B)"][index],
        "val_mAP50_at_best": results["metrics/mAP50(B)"][index],
        "val_mAP50_95_at_best": fitness[index],
    }
    if results.get("time"):
        cumulative = np.asarray(results["time"], dtype=float)
        per_epoch = np.diff(np.concatenate([[0.0], cumulative]))
        # The 'time' column restarts after a resumption: the first epoch of the new
        # session is then measured from that session's start.
        per_epoch = np.where(per_epoch < 0, cumulative, per_epoch)
        info["train_time_s"] = float(per_epoch.sum())
        # The first epoch includes start-up costs; exclude it when possible.
        info["median_epoch_s"] = float(np.median(per_epoch[1:] if len(per_epoch) > 1 else per_epoch))
    return info


def test_summary(metrics) -> dict[str, object]:
    """Overall and per-class test metrics plus the maximum-F1 operating point."""
    from ultralytics.utils.metrics import smooth

    box = metrics.box
    names = metrics.names
    px = np.asarray(box.px)
    f1_curve = np.asarray(box.f1_curve)
    best = int(smooth(f1_curve.mean(0), 0.1).argmax())  # Ultralytics operating point for P and R
    per_class = []
    for i, class_index in enumerate(box.ap_class_index):
        per_class.append(
            {
                "class": names[int(class_index)],
                "precision": float(box.p[i]),
                "recall": float(box.r[i]),
                "f1": float(box.f1[i]),
                "AP50": float(box.ap50[i]),
                "AP50_95": float(box.ap[i]),
            }
        )
    return {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
        "operating_conf": float(px[best]),
        "mean_f1_at_operating_conf": float(f1_curve.mean(0)[best]),
        "per_class": per_class,
    }


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", required=True, choices=sorted(FREEZE))
    parser.add_argument("--model", required=True, help="Architecture YAML, or yolo12n.pt for the baseline.")
    parser.add_argument("--data", required=True, help="Fold data YAML with train/val/test keys.")
    parser.add_argument("--fold-info", required=True, help="JSON written by the notebook for this fold.")
    parser.add_argument("--out", required=True, help="Output directory of this run.")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--epochs", type=int, default=int(PROTOCOL["epochs"]))
    parser.add_argument("--resume", action="store_true", help="Continue from train/weights/last.pt.")
    args = parser.parse_args()

    import torch
    import torchvision
    import ultralytics
    from ultralytics import YOLO

    out = Path(args.out)
    train_dir = out / "train"
    last_pt = train_dir / "weights" / "last.pt"
    resume_log_path = out / "resume_log.json"
    resume_log = json.loads(resume_log_path.read_text(encoding="utf-8")) if resume_log_path.exists() else []

    if args.resume:
        if not last_pt.is_file():
            raise FileNotFoundError(f"--resume requires {last_pt}")
    else:
        out.mkdir(parents=True, exist_ok=False)
        fold_info = Path(args.fold_info)
        manifest = fold_info.with_name("fold_manifest.csv")
        if manifest.is_file():
            shutil.copy2(manifest, out / manifest.name)  # image-level record of this fold's partition
    fold = json.loads(Path(args.fold_info).read_text(encoding="utf-8"))

    start = time.time()
    if not args.resume:
        model = YOLO(args.model, task="detect")
        train_args = dict(PROTOCOL)
        train_args.update(
            data=args.data,
            epochs=args.epochs,
            seed=args.seed,
            device="0",
            project=str(out),
            name="train",
            exist_ok=False,
            freeze=FREEZE[args.config],
        )
        if args.epochs <= int(PROTOCOL["close_mosaic"]):
            # Smoke test only: keep mosaic active, as in full runs (which all stop early,
            # before the last 10 epochs), so that the measured epoch time is representative.
            train_args["close_mosaic"] = 0
        model.train(**train_args)
    else:
        ckpt = torch.load(last_pt, map_location="cpu", weights_only=False)
        if ckpt.get("optimizer") is None:
            # Training had finished (Ultralytics strips the final last.pt); only the
            # test evaluation is missing.
            resume_log.append({"time": now(), "action": "test evaluation only (training already finished)"})
        else:
            last_epoch = int(ckpt["epoch"]) + 1  # results.csv numbers epochs from 1
            kept = truncate_results_csv(train_dir / "results.csv", last_epoch)
            history = read_results_csv(train_dir / "results.csv")
            best_epoch = int(history["epoch"][int(np.argmax(history[FITNESS_KEY]))])
            best_fitness = float(ckpt["best_fitness"])
            if kept != last_epoch:
                raise RuntimeError(f"results.csv has {kept} complete rows but last.pt is at epoch {last_epoch}.")

            def restore_early_stopping(trainer) -> None:
                trainer.stopper.best_fitness = best_fitness
                trainer.stopper.best_epoch = best_epoch

            resume_log.append(
                {
                    "time": now(),
                    "action": "resumed",
                    "completed_epochs_before_resume": last_epoch,
                    "best_epoch_before_resume": best_epoch,
                    "best_fitness_before_resume": best_fitness,
                }
            )
            resume_log_path.write_text(json.dumps(resume_log, indent=2), encoding="utf-8")
            model = YOLO(str(last_pt))
            model.add_callback("on_train_start", restore_early_stopping)
            model.train(resume=str(last_pt), device="0")
    session_time = time.time() - start
    if resume_log:
        resume_log_path.write_text(json.dumps(resume_log, indent=2), encoding="utf-8")

    best_pt = train_dir / "weights" / "best.pt"
    selection = selection_info(read_results_csv(train_dir / "results.csv"))

    test_model = YOLO(str(best_pt), task="detect")
    # Unfused trained model (3 classes). YOLO.info(verbose=False) returns None in 8.4.104.
    n_params = sum(p.numel() for p in test_model.model.parameters())
    metrics = test_model.val(
        data=args.data,
        split="test",
        device="0",
        project=str(out),
        name="test_eval",
        exist_ok=True,
        **TEST_EVAL,
    )

    curves = metrics.box.curves_results
    np.savez_compressed(
        out / "test_curves.npz",
        px=np.asarray(curves[0][0]),
        precision_recall=np.asarray(curves[0][1]),
        f1_confidence=np.asarray(curves[1][1]),
        precision_confidence=np.asarray(curves[2][1]),
        recall_confidence=np.asarray(curves[3][1]),
        class_index=np.asarray(metrics.box.ap_class_index),
    )
    np.save(out / "test_confusion_matrix.npy", np.asarray(metrics.confusion_matrix.matrix))

    summary = {
        "config": args.config,
        "seed": args.seed,
        "epochs_max": args.epochs,
        **fold,
        "params_unfused": int(n_params),
        "selection": selection,
        "last_session_time_s": session_time,
        "resumptions": resume_log,
        "test": test_summary(metrics),
        "files": {
            "best_pt": "train/weights/best.pt",
            "args_yaml": "train/args.yaml",
            "results_csv": "train/results.csv",
            "test_predictions_json": "test_eval/predictions.json",
            "test_curves": "test_curves.npz",
            "test_confusion_matrix": "test_confusion_matrix.npy",
        },
        "environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "torchvision": torchvision.__version__,
            "ultralytics": ultralytics.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
    }
    # Written last: its presence marks a completed run.
    (out / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary["test"], indent=2))


if __name__ == "__main__":
    main()

## 6. Pre-download shared assets (avoids download races between parallel runs)

In [ ]:
os.chdir(WORK)
from ultralytics.utils.checks import check_font
from ultralytics.utils.downloads import attempt_download_asset

torchvision.models.get_model("mobilenet_v3_small", weights="DEFAULT")  # ImageNet weights -> torch hub cache
check_font("Arial.ttf")
attempt_download_asset("yolo26n.pt")  # used by the Ultralytics AMP check
if CONFIG == "yolov12n":
    attempt_download_asset("yolo12n.pt")
MODEL_SOURCE = str(WORK / "yolo12n.pt") if CONFIG == "yolov12n" else str(MODEL_YAML)
assert Path(MODEL_SOURCE).is_file(), MODEL_SOURCE
print("Model source:", MODEL_SOURCE)
if CONFIG in ABLATIONS:  # same parameters as measured offline, and the same (non-legacy) detection head type
    from ultralytics import YOLO
    from ultralytics.nn.modules.head import Detect
    _m = YOLO(MODEL_SOURCE, task="detect").model
    _n, _h = sum(p.numel() for p in _m.parameters()), sum(p.numel() for p in _m.model[-1].parameters())
    print(f"{CONFIG}: {_n:,} parameters (adopted 746,015), detection head {_h:,} (adopted 265,449)")
    assert (_n, _h) == EXPECTED[CONFIG] and Detect.legacy is False, (_n, _h, Detect.legacy)
    del _m


## 7. Persistent store (private Kaggle dataset)

In [ ]:
import hashlib, torch, zipfile
from concurrent.futures import ThreadPoolExecutor

SEP = "__"                          # flattened file names: <run>__train__weights__last.pt
MANIFEST = "store_manifest.json"    # list of the files of each version, uploaded with it
KAGGLE_CLI_VERSION = "2.2.2"
MODE = "storetest" if STORE_SELF_TEST else ("smoke" if SMOKE_TEST else "runs")
RUN_ROOT = WORK / f"cv_{MODE}" / CONFIG
if MODE != "runs" and RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)  # smoke tests and store self-tests always start from scratch
RUN_ROOT.mkdir(parents=True, exist_ok=True)
STORE_SLUG = "cv5fold-storetest" if STORE_SELF_TEST else f"cv5fold-{MODE}-{CONFIG}-seed{SEED}"
STAGE = WORK / "store_stage"
STORE_EXISTS = [False]  # set once the store dataset is known to exist
LAST_UPLOADED = [None]  # fingerprint of the files in the last uploaded version

if USE_STORE:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"kaggle=={KAGGLE_CLI_VERSION}"])
    STORE_ID = f"{os.environ['KAGGLE_USERNAME']}/{STORE_SLUG}"


def kaggle_cli(*args, timeout=1800):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr).strip()


def store_status():
    code, text = kaggle_cli("datasets", "status", STORE_ID, timeout=120)
    low = text.lower()
    if code != 0 or "404" in low or "not found" in low or "forbidden" in low:
        return None
    return low


def wait_ready(max_s):
    t0 = time.time()
    while time.time() - t0 < max_s:
        status = store_status()
        if status is not None and "ready" in status:
            return True
        time.sleep(15)
    return False


def store_files():
    # Completed runs: results, checkpoint and test outputs (no periodic checkpoints, last.pt,
    # image mosaics or validation-fold plots). Running runs: what is needed to resume
    # (best.pt is listed before last.pt on purpose).
    files = []
    for run in sorted(p for p in RUN_ROOT.glob("test*_seed*") if p.is_dir()):
        if (run / "summary.json").exists():
            for p in sorted(run.rglob("*")):
                if not p.is_file() or p.suffix == ".jpg":
                    continue
                if p.parent.name == "weights" and (p.name.startswith("epoch") or p.name == "last.pt"):
                    continue
                if p.suffix == ".png" and p.parent.name == "train" and p.name != "results.png":
                    continue
                files.append(p)
        else:
            for rel in ("train/weights/best.pt", "train/weights/last.pt", "train/results.csv",
                        "train/args.yaml", "resume_log.json", "fold_manifest.csv"):
                if (run / rel).is_file():
                    files.append(run / rel)
    return files + sorted(RUN_ROOT.glob("log_test*.txt"))


def stage_consistent():
    # For every unfinished run: last.pt and best.pt must load, describe the same best fitness,
    # and results.csv must already contain the epoch stored in last.pt.
    for last_copy in STAGE.glob(f"*{SEP}train{SEP}weights{SEP}last.pt"):
        run = last_copy.name.split(SEP)[0]
        if (STAGE / f"{run}{SEP}summary.json").exists():
            continue
        try:
            last = torch.load(last_copy, map_location="cpu", weights_only=False)
            if last.get("optimizer") is None:
                continue  # training finished; final checkpoints are stable
            best = torch.load(STAGE / f"{run}{SEP}train{SEP}weights{SEP}best.pt", map_location="cpu", weights_only=False)
            rows = (STAGE / f"{run}{SEP}train{SEP}results.csv").read_text().splitlines()[1:]
        except Exception:
            return False
        if float(best["best_fitness"]) != float(last["best_fitness"]) or best["epoch"] > last["epoch"]:
            return False
        if len(rows) < int(last["epoch"]) + 1:
            return False
    return True


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def snapshot(note):
    if not USE_STORE:
        return False
    files = store_files()
    if not files:
        return False
    fingerprint = tuple((str(p.relative_to(RUN_ROOT)), p.stat().st_size, p.stat().st_mtime_ns) for p in files)
    if fingerprint == LAST_UPLOADED[0]:
        return True  # nothing changed since the last uploaded version
    for _ in range(3):
        if STAGE.exists():
            shutil.rmtree(STAGE)
        STAGE.mkdir()
        for p in files:
            rel = p.relative_to(RUN_ROOT)
            assert all(SEP not in part for part in rel.parts), rel
            shutil.copy2(p, STAGE / SEP.join(rel.parts))
        if stage_consistent():
            break
        time.sleep(15)  # a checkpoint was being written: copy again
    else:
        print(f"[{time.strftime('%H:%M')}] snapshot postponed (checkpoint being written)")
        return False
    manifest = {f.name: {"size": f.stat().st_size, "sha256": sha256(f)} for f in sorted(STAGE.iterdir())}
    (STAGE / MANIFEST).write_text(json.dumps(manifest, indent=1))
    (STAGE / "dataset-metadata.json").write_text(json.dumps(
        {"title": STORE_SLUG, "id": STORE_ID, "licenses": [{"name": "CC0-1.0"}]}))
    if not STORE_EXISTS[0]:
        STORE_EXISTS[0] = store_status() is not None
    if not STORE_EXISTS[0]:
        code, text = kaggle_cli("datasets", "create", "-p", str(STAGE), "-t", "-q")
        STORE_EXISTS[0] = code == 0  # status may lag behind creation; never create twice
    else:
        wait_ready(300)  # the previous version must be processed first
        code, text = kaggle_cli("datasets", "version", "-p", str(STAGE), "-m", note[:200], "-t", "-q")
    if code == 0:
        LAST_UPLOADED[0] = fingerprint
    size = sum(m["size"] for m in manifest.values()) / 2**20
    status = "OK" if code == 0 else "FAILED"
    print(f"[{time.strftime('%H:%M')}] snapshot {status}: {len(files)} files, {size:.1f} MiB — {note}")
    if code != 0:
        print(text[-1500:])
    return code == 0


def download_one(name, dest):
    # Single-file download (the whole-dataset download endpoint answers 404 for these datasets).
    target, archive = dest / name, dest / f"{name}.zip"
    for p in (target, archive):
        if p.exists():
            p.unlink()
    kaggle_cli("datasets", "download", STORE_ID, "-f", name, "-p", str(dest), "-o", "-q", timeout=900)
    if not target.exists() and archive.exists():  # large files may be served zipped
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(dest)
        archive.unlink()
    return target if target.exists() else None


def download_store(dest, expected=None, manifest_attempts=5, wait_s=30):
    # Downloads the latest version file by file and checks every file against the manifest
    # (SHA-256): while a new version propagates, Kaggle may serve some files from the previous
    # one, and such a mix is detected and downloaded again. Returns {name: path}, or None if
    # `expected` is given and Kaggle does not serve that version yet.
    for cycle in range(3):
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True)
        manifest = None
        for attempt in range(manifest_attempts):
            f = download_one(MANIFEST, dest)
            if f is not None:
                manifest = json.loads(f.read_text())
                if expected is None or manifest == expected:
                    break
            if attempt < manifest_attempts - 1:
                time.sleep(wait_s)
        if manifest is None:
            raise RuntimeError(f"Could not download {MANIFEST} from {STORE_ID}.")
        if expected is not None and manifest != expected:
            return None
        intact = lambda name, path: path is not None and sha256(path) == manifest[name]["sha256"]
        with ThreadPoolExecutor(4) as pool:
            paths = dict(zip(manifest, pool.map(lambda n: download_one(n, dest), manifest)))
        for _ in range(2):
            bad = [n for n, p in paths.items() if not intact(n, p)]
            if not bad:
                return paths
            time.sleep(wait_s)
            for name in bad:
                paths[name] = download_one(name, dest)
        if all(intact(n, p) for n, p in paths.items()):
            return paths
        time.sleep(wait_s)  # a newer version is probably propagating: start again from its manifest
    raise RuntimeError(f"Files of {STORE_ID} do not match its manifest after 3 attempts.")


def restore(expected=None, manifest_attempts=5):
    if not USE_STORE:
        return None
    if store_status() is None:
        print(f"No store {STORE_ID} yet: fresh start.")
        return {}
    wait_ready(600)
    paths = download_store(WORK / "store_restore", expected, manifest_attempts)
    if paths is None:
        return None
    restored = 0
    for name, f in paths.items():
        dst = RUN_ROOT.joinpath(*name.split(SEP))
        if not dst.exists():  # files already present in this session are newer
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dst)
            restored += 1
    print(f"Restored {restored} files from {STORE_ID}.")
    return paths


if USE_STORE:
    print("Store:", STORE_ID, "| status:", store_status() or "not created yet")

### Store self-test (CPU only, no GPU quota)

In [ ]:
# Runs only with STORE_SELF_TEST = True: builds a small run tree (one completed run with a
# 12 MiB file, one interrupted run), uploads it, wipes it, restores it and compares.
if STORE_SELF_TEST:
    done = RUN_ROOT / "test1_seed42"
    (done / "train" / "weights").mkdir(parents=True)
    (done / "test_eval").mkdir()
    (done / "summary.json").write_text(json.dumps({"self_test": True}))
    (done / "train" / "results.csv").write_text("epoch,time,metrics/mAP50-95(B)\n1,30,0.1\n")
    (done / "test_eval" / "predictions.json").write_bytes(os.urandom(12 * 2**20))
    torch.save({"epoch": -1, "best_fitness": None, "optimizer": None}, done / "train" / "weights" / "best.pt")
    run = RUN_ROOT / "test2_seed42"
    (run / "train" / "weights").mkdir(parents=True)
    for name in ("best.pt", "last.pt"):
        torch.save({"epoch": 1, "best_fitness": 0.5, "optimizer": {"state": {}}}, run / "train" / "weights" / name)
    (run / "train" / "results.csv").write_text("epoch,time,metrics/mAP50-95(B)\n1,30,0.4\n2,60,0.5\n")
    (run / "train" / "args.yaml").write_text("epochs: 400\n")
    (RUN_ROOT / "log_test2_seed42.txt").write_text("self-test\n")

    assert snapshot("store self-test"), "Upload failed: see the message above."
    local = {str(p.relative_to(RUN_ROOT)): p.stat().st_size for p in store_files()}
    expected = json.loads((STAGE / MANIFEST).read_text())
    shutil.rmtree(RUN_ROOT)
    RUN_ROOT.mkdir(parents=True)
    print("Waiting for Kaggle to serve the new version (up to 10 min)...")
    if restore(expected=expected, manifest_attempts=20) is None:
        print("STORE SELF-TEST INCONCLUSIVE: the new version was not served within 10 min; run this cell again later.")
    else:
        restored = {str(p.relative_to(RUN_ROOT)): p.stat().st_size for p in RUN_ROOT.rglob("*") if p.is_file()}
        print("STORE SELF-TEST OK: upload and file-by-file restore are identical." if restored == local
              else f"STORE SELF-TEST FAILED: differences {sorted(set(local.items()) ^ set(restored.items()))[:5]}")

## 8. Train and evaluate (one fold per GPU), with snapshots and automatic resumption

In [ ]:
if STORE_SELF_TEST:
    raise RuntimeError("STORE_SELF_TEST = True: training is skipped on purpose. Set it back to False to train.")
folds = ([1, 2] if N_GPU >= 2 else [1]) if SMOKE_TEST else list(TEST_FOLDS)
epochs = 4 if SMOKE_TEST else 400
poll_s = 5 if SMOKE_TEST else 30

if not SMOKE_TEST and restore() is None and USE_STORE:
    raise RuntimeError("The store could not be restored: training not started, to avoid redoing completed work.")

jobs = []
for k in folds:
    out = RUN_ROOT / f"test{k}_seed{SEED}"
    if (out / "summary.json").exists():
        print(f"test fold {k}: completed, skipped")
        continue
    resume = (out / "train" / "weights" / "last.pt").is_file()
    if out.exists() and not resume:
        shutil.rmtree(out)  # interrupted before its first checkpoint: restart from scratch
    data_yaml, info_path, info = build_fold(k)
    cmd = [sys.executable, str(WORK / "train_fold.py"), "--config", CONFIG, "--model", MODEL_SOURCE,
           "--data", str(data_yaml), "--fold-info", str(info_path), "--out", str(out),
           "--seed", str(SEED), "--epochs", str(epochs)]
    if resume:
        cmd.append("--resume")
        print(f"test fold {k}: resuming from its last checkpoint")
    jobs.append({"k": k, "out": out, "cmd": cmd, "log": RUN_ROOT / f"log_test{k}_seed{SEED}.txt"})


def epochs_done(out):
    csv_path = out / "train" / "results.csv"
    return max(0, len(csv_path.read_text().splitlines()) - 1) if csv_path.exists() else 0


slots = max(1, min(N_GPU, len(jobs)))
queue, running, failed = list(jobs), {}, []
t_start = last_report = last_snapshot = time.time()
try:
    while queue or running:
        for gpu in range(slots):
            if gpu not in running and queue:
                job = queue.pop(0)
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                handle = open(job["log"], "a")
                handle.write(f"\n===== session started {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
                handle.flush()
                proc = subprocess.Popen(job["cmd"], env=env, stdout=handle, stderr=subprocess.STDOUT, cwd=WORK)
                running[gpu] = (proc, job, handle)
                print(f"[{time.strftime('%H:%M')}] test fold {job['k']} started on GPU {gpu}")
        time.sleep(poll_s)
        if SMOKE_TEST:
            # Resumption test: interrupt the first fold once after 2 epochs (when last.pt is fully
            # written), then relaunch it with --resume, exactly as after a real interruption.
            for gpu, (proc, job, handle) in list(running.items()):
                last_pt = job["out"] / "train" / "weights" / "last.pt"
                if (job["k"] == folds[0] and not job.get("interrupted") and epochs_done(job["out"]) >= 2
                        and last_pt.exists() and time.time() - last_pt.stat().st_mtime > 3):
                    proc.terminate()
                    proc.wait()
                    handle.close()
                    del running[gpu]
                    job["interrupted"] = True
                    job["cmd"] = job["cmd"] + ["--resume"]
                    queue.insert(0, job)
                    print(f"[{time.strftime('%H:%M')}] smoke test: fold {job['k']} interrupted on purpose, relaunched with --resume")
        fold_finished = False
        for gpu in list(running):
            proc, job, handle = running[gpu]
            if proc.poll() is not None:
                handle.close()
                del running[gpu]
                fold_finished = True
                status = "done" if proc.returncode == 0 and (job["out"] / "summary.json").exists() else "FAILED"
                print(f"[{time.strftime('%H:%M')}] test fold {job['k']}: {status} (code {proc.returncode})")
                if status == "FAILED":
                    failed.append(job)
                    print("".join(job["log"].read_text(errors="replace").splitlines(True)[-40:]))
        state = ", ".join(f"fold {j['k']}: epoch {epochs_done(j['out'])}" for _, j, _ in running.values())
        if USE_STORE and (fold_finished or time.time() - last_snapshot > SNAPSHOT_MINUTES * 60):
            if snapshot(f"{CONFIG} seed {SEED} — {state or 'folds finished'}"):
                last_snapshot = time.time()
        if running and time.time() - last_report > 900:
            last_report = time.time()
            print(f"[{time.strftime('%H:%M')}] {(time.time() - t_start) / 3600:.2f} h elapsed — {state}")
finally:
    if USE_STORE and not snapshot(f"{CONFIG} seed {SEED} — end of session"):
        print("Final snapshot failed: see the message above (secrets attached? Internet on?).")
assert not failed, f"Failed folds: {[j['k'] for j in failed]}"
print(f"Session finished in {(time.time() - t_start) / 3600:.2f} h")

## 9. Summary of the completed folds

In [ ]:
rows = []
for path in sorted(RUN_ROOT.glob("test*_seed*/summary.json")):
    s = json.loads(path.read_text())
    t, sel = s["test"], s["selection"]
    glioma = next(c for c in t["per_class"] if c["class"] == "glioma")
    rows.append({"test_fold": s["test_fold"], "val_fold": s["val_fold"], "best_epoch": sel["best_epoch"],
                 "epochs_run": sel["epochs_completed"], "s_per_epoch": round(sel.get("median_epoch_s", float("nan")), 1),
                 "resumed": len([r for r in s.get("resumptions", []) if r.get("action") == "resumed"]),
                 "P": t["precision"], "R": t["recall"], "mAP50": t["mAP50"], "mAP50_95": t["mAP50_95"],
                 "glioma_R": glioma["recall"], "glioma_AP50": glioma["AP50"]})
table = pd.DataFrame(rows).round(3)
print(table.to_string(index=False))
table.to_csv(RUN_ROOT / f"summary_{CONFIG}.csv", index=False)
if SMOKE_TEST and len(table):
    s = float(table["s_per_epoch"].median())
    print(f"\nSmoke test: {s:.1f} s/epoch with {len(table)} run(s) in parallel.")
    print("Resumption test:", "OK" if table["resumed"].max() >= 1 else "NOT EXERCISED (check the log)")
    for n in (200, 300, 400):
        print(f"  a {n}-epoch run would take about {s * n / 3600:.1f} h")
    print("Test metrics after 4 epochs are meaningless; only the pipeline and the timing are checked.")

## 10. Archive for download (periodic epoch checkpoints excluded)

In [ ]:
import zipfile
archive = WORK / f"{RUN_ROOT.parent.name}_{CONFIG}_seed{SEED}_folds{''.join(str(k) for k in folds)}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and not (path.parent.name == "weights" and path.name.startswith("epoch")):
            zf.write(path, path.relative_to(RUN_ROOT.parent))
    zf.writestr("environment.json", json.dumps(ENV, indent=2))
print(f"{archive} ({archive.stat().st_size / 2**20:.1f} MiB)")